# Preprocess and Extract Dataset Shape

Read the preprocessed THINGS-EEG2 EEG tensors and extracted CLIP feature tensors, then print detailed nested structure information.

In [ ]:
import os
import sys
from pathlib import Path

import torch
from omegaconf import OmegaConf
from rich.console import Console
from rich.panel import Panel
from rich.pretty import Pretty
from rich.table import Table
from rich.tree import Tree

console = Console()

In [ ]:
def find_project_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "configs").exists() and (candidate / "src").exists():
            return candidate
    return start


def load_paths_config(root):
    os.environ.setdefault("PROJECT_ROOT", str(root))
    paths = OmegaConf.load(root / "configs" / "paths" / "default.yaml")
    return OmegaConf.create({"paths": paths})


def config_path(paths_config, key):
    value = OmegaConf.select(paths_config, f"paths.{key}")
    return Path(str(value)).expanduser()


def subject_name(subject_id):
    return f"sub-{int(subject_id):02d}"


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.components.thingseeg2_dataset import ThingsEEG2Dataset
from src.data.thingseeg2_datamodule import ThingsEEG2DataModule
from src.utils.clip import resolve_clip_model_id, sanitize_clip_model_name

paths_config = load_paths_config(ROOT)
preprocess_config = OmegaConf.load(ROOT / "configs" / "preprocess" / "thingseeg2.yaml")
extract_config = OmegaConf.load(ROOT / "configs" / "extract" / "thingseeg2.yaml")

subject = subject_name(preprocess_config.subject_id)
model_name, model_id = resolve_clip_model_id(
    extract_config.model_name,
    extract_config.model_id,
)
feature_mode = extract_config.feature_mode
partitions = list(extract_config.partitions)

eeg_root = config_path(paths_config, "thingseeg2_preprocessed_dir")
clip_root = config_path(paths_config, "thingseeg2_clip_features_dir")
eeg_dir = eeg_root / subject
clip_dir = (
    clip_root
    / sanitize_clip_model_name(model_name)
    / sanitize_clip_model_name(model_id)
    / feature_mode
)

DATASETS = {f"eeg_{partition}": eeg_dir / f"{partition}.pt" for partition in partitions} | {
    f"clip_{partition}": clip_dir / f"{partition}.pt" for partition in partitions
}

console.print(Panel.fit(str(ROOT), title="Project Root", border_style="cyan"))

config_table = Table(title="Resolved Config", show_lines=True)
config_table.add_column("Field", style="bold cyan")
config_table.add_column("Value", overflow="fold")
config_table.add_row("subject", subject)
config_table.add_row("model_name", model_name)
config_table.add_row("model_id", model_id)
config_table.add_row("feature_mode", feature_mode)
config_table.add_row("partitions", ", ".join(partitions))
config_table.add_row("eeg_dir", str(eeg_dir))
config_table.add_row("clip_dir", str(clip_dir))
console.print(config_table)

path_table = Table(title="Dataset Files", show_lines=True)
path_table.add_column("Name", style="bold cyan")
path_table.add_column("Path", overflow="fold")
path_table.add_column("Exists", justify="center")
path_table.add_column("Size", justify="right")

for name, path in DATASETS.items():
    exists = path.exists()
    size = f"{path.stat().st_size:,} bytes" if exists else "-"
    path_table.add_row(name, str(path), "yes" if exists else "no", size)

console.print(path_table)

In [ ]:
def tensor_summary(value):
    summary = {
        "shape": tuple(value.shape),
        "dtype": str(value.dtype),
        "device": str(value.device),
        "numel": f"{value.numel():,}",
    }

    if value.numel() > 0 and value.dtype.is_floating_point:
        value_float = value.float()
        summary.update(
            {
                "finite": bool(torch.isfinite(value).all().item()),
                "mean": f"{value_float.mean().item():.6g}",
                "std": f"{value_float.std(unbiased=False).item():.6g}",
                "min": f"{value_float.min().item():.6g}",
                "max": f"{value_float.max().item():.6g}",
            }
        )
    elif value.numel() > 0:
        summary.update({"min": value.min().item(), "max": value.max().item()})

    return summary


def add_value_to_tree(tree, name, value, max_items=5):
    if torch.is_tensor(value):
        branch = tree.add(f"[bold green]{name}[/]: Tensor")
        for key, item in tensor_summary(value).items():
            branch.add(f"[cyan]{key}[/]: {item}")
        return

    if isinstance(value, dict):
        branch = tree.add(f"[bold blue]{name}[/]: dict keys={list(value.keys())}")
        for key, item in value.items():
            add_value_to_tree(branch, str(key), item, max_items=max_items)
        return

    if isinstance(value, (list, tuple)):
        branch = tree.add(f"[bold magenta]{name}[/]: {type(value).__name__} len={len(value)}")
        for idx, item in enumerate(value[:max_items]):
            add_value_to_tree(branch, f"[{idx}]", item, max_items=max_items)
        if len(value) > max_items:
            branch.add(f"... {len(value) - max_items} more")
        return

    tree.add(f"[bold]{name}[/]: {type(value).__name__} {value!r}")


def describe_value(value, name="root", max_items=5):
    tree = Tree(f"[bold]{name}[/]")
    add_value_to_tree(tree, name, value, max_items=max_items)
    console.print(tree)

In [ ]:
loaded = {}

for name, path in DATASETS.items():
    console.rule(f"[bold cyan]{name}")
    console.print(f"[dim]{path}[/]")

    loaded[name] = torch.load(path, map_location="cpu", weights_only=False)
    describe_value(loaded[name], name=name)

In [ ]:
summary = {}

for name, data in loaded.items():
    if isinstance(data, dict):
        summary[name] = {
            key: {
                "type": type(value).__name__,
                "shape": tuple(value.shape) if torch.is_tensor(value) else None,
                "dtype": str(value.dtype) if torch.is_tensor(value) else None,
                "len": (
                    len(value)
                    if hasattr(value, "__len__") and not torch.is_tensor(value)
                    else None
                ),
            }
            for key, value in data.items()
        }
    else:
        summary[name] = {
            "type": type(data).__name__,
            "shape": tuple(data.shape) if torch.is_tensor(data) else None,
            "dtype": str(data.dtype) if torch.is_tensor(data) else None,
        }

console.print(Panel(Pretty(summary), title="Summary", border_style="green"))

In [ ]:
def format_value(value):
    if isinstance(value, dict):
        return Pretty(value)
    if isinstance(value, (list, tuple)):
        return ", ".join(str(item) for item in value)
    return str(value)


def sample_field_summary(value):
    if torch.is_tensor(value):
        return {
            "type": "Tensor",
            "shape": tuple(value.shape),
            "dtype": str(value.dtype),
            "value": value.item() if value.ndim == 0 else "",
        }
    return {
        "type": type(value).__name__,
        "shape": "",
        "dtype": "",
        "value": value,
    }


def render_mapping_table(title, mapping, field_name="Field"):
    table = Table(title=title, show_lines=True)
    table.add_column(field_name, style="bold cyan")
    table.add_column("Value", overflow="fold")
    for key, value in mapping.items():
        table.add_row(str(key), format_value(value))
    console.print(table)


def render_dataset_summary(name, dataset):
    render_mapping_table(f"{name} Dataset Summary", dataset.describe())

    sample = dataset[0]
    sample_table = Table(title=f"{name} First Sample", show_lines=True)
    sample_table.add_column("Key", style="bold cyan")
    sample_table.add_column("Type")
    sample_table.add_column("Shape")
    sample_table.add_column("DType")
    sample_table.add_column("Value", overflow="fold")
    for key, value in sample.items():
        item = sample_field_summary(value)
        sample_table.add_row(
            key,
            item["type"],
            format_value(item["shape"]),
            item["dtype"],
            str(item["value"]),
        )
    console.print(sample_table)


def render_datamodule_description(description):
    console.print(
        Panel(
            Pretty(description), title="ThingsEEG2DataModule Description", border_style="magenta"
        )
    )

    render_mapping_table(
        "DataModule Overview",
        {
            "experiment_setting": description["experiment_setting"],
            "subjects": description["subjects"],
            "paths": description["paths"],
            "features": description["features"],
            "split": description["split"],
            "dataset_options": description["dataset_options"],
        },
    )

    dataset_table = Table(title="DataModule Dataset Splits", show_lines=True)
    dataset_table.add_column("Split", style="bold cyan")
    dataset_table.add_column("Type")
    dataset_table.add_column("Length", justify="right")
    dataset_table.add_column("Source", overflow="fold")
    for split, summary in description["datasets"].items():
        if summary is None:
            dataset_table.add_row(split, "", "", "not loaded")
            continue
        dataset_table.add_row(
            split,
            summary["type"],
            str(summary["length"]),
            str(summary.get("source", {})),
        )
    console.print(dataset_table)

    batch_table = Table(title="DataModule First Batch", show_lines=True)
    batch_table.add_column("Split", style="bold cyan")
    batch_table.add_column("Key")
    batch_table.add_column("Type")
    batch_table.add_column("Shape")
    batch_table.add_column("DType")
    for split, batch in description.get("sample_batches", {}).items():
        if batch is None:
            batch_table.add_row(split, "", "", "", "")
            continue
        for key, value in batch.items():
            batch_table.add_row(
                split,
                key,
                value.get("type", ""),
                format_value(value.get("shape", "")),
                value.get("dtype", ""),
            )
    console.print(batch_table)


thingseeg2_datasets = {}
for partition in partitions:
    console.rule(f"[bold yellow]ThingsEEG2Dataset {partition}")
    dataset = ThingsEEG2Dataset(
        eeg_data_dir=eeg_root,
        clip_features_dir=clip_root,
        subjects=subject,
        partition=partition,
        average_reps=True,
        model_name=model_name,
        model_id=model_id,
        feature_mode=feature_mode,
    )
    thingseeg2_datasets[partition] = dataset
    render_dataset_summary(partition, dataset)

console.rule("[bold magenta]ThingsEEG2DataModule")
datamodule = ThingsEEG2DataModule(
    eeg_data_dir=eeg_root,
    clip_features_dir=clip_root,
    subjects=subject,
    experiment_setting="cross-subject",
    train_val_split=[0.95, 0.05],
    train_batch_size=256,
    val_batch_size=200,
    test_batch_size=200,
    num_workers=0,
    average_reps=True,
    model_name=model_name,
    model_id=model_id,
    feature_mode=feature_mode,
)
datamodule.setup()
render_datamodule_description(datamodule.describe(include_batch=True))